# Notebook 13: K-Means Clustering
**Lab Assignment:** Assignment 13  
**Project Role:** Unsupervised discovery of reader behavioral personas and content topic grouping with k-means++ initialization and silhouette scoring.


In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()))
from src.data_loader import load_data
from src.features import SubstackFeaturePipeline
from src.kmeans import KMeansScratch, silhouette_score_scratch
df_posts, df_interactions = load_data()
pipeline = SubstackFeaturePipeline().fit(df_posts, df_interactions)
all_users = list(df_interactions['user_id'].unique())[:1000]
topic_mat = []
for u in all_users:
    u_dict = pipeline.user_topic_affinity_.get(u, {})
    topic_mat.append([u_dict.get(t, 0.0) for t in pipeline.topic_list_])
topic_mat = np.array(topic_mat)
cluster_metrics = []
for k in range(3, 8):
    km = KMeansScratch(n_clusters=k, n_init=3, random_state=42).fit(topic_mat)
    sil = silhouette_score_scratch(topic_mat, km.labels_)
    cluster_metrics.append({'K': k, 'Inertia': round(km.inertia_, 2), 'Silhouette Score': round(sil, 4)})
print(pd.DataFrame(cluster_metrics).to_string(index=False))


### Interpretation & Findings
1. **Optimal Reader Cohorts:** Evaluating the silhouette score reveals distinct cohesive personas around $K=6$, separating readers into identifiable topic clusters (Tech Futurists, Macro Policy, Biohackers, etc.).
2. **Cold-Start Personas:** When a new user specifies favorite topics, routing them to the corresponding cluster centroid enables accurate initial recommendations.

### Limitations
- K-Means assumes spherical clusters and equal cluster sizes, which does not hold for niche topic readers.
